# LeetCode 3127 — Make a Square with the Same Color bằng Python

Link đề: https://leetcode.com/problems/make-a-square-with-the-same-color/ · Độ khó: **Easy** · Chủ đề: Array, Matrix, Enumeration

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def canMakeSquare(self, grid: List[List[str]]) -> bool:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho lưới `3 × 3` các ô màu `'B'` (đen) hoặc `'W'` (trắng); được **đổi màu tối đa một ô**; hỏi có thể tạo ra một **hình vuông `2 × 2` cùng màu** hay không.
2. Nắm cách làm việc với **ma trận 2 chiều** (`grid[i][j]`), cách lấy **bốn ô của một hình vuông `2 × 2`** từ góc trên-trái `(r, c)`, **sao chép sâu** để thử nghiệm mà không làm hỏng lưới gốc, và `list.count`.
3. Đi từ cách **thử mọi phép đổi màu (hoặc không đổi) rồi kiểm tra có hình vuông cùng màu** → **với mỗi hình vuông `2 × 2`, đếm số ô `B`** → **dùng số lần xuất hiện nhiều nhất của một màu** → **viết gọn bằng `any`/hàm hỗ trợ**.
4. Hiểu mấu chốt: một hình vuông `2 × 2` có `4` ô; nó có thể thành cùng màu bằng **tối đa 1 lần đổi** khi **một màu chiếm `>= 3` ô** (đã có cùng màu: `4`; hoặc `3-1`: đổi ô lẻ); còn **`2-2`** thì cần `2` lần đổi nên không được.
5. Tránh các lỗi kinh điển: chỉ xét trường hợp đúng `3` ô cùng màu (bỏ sót `4`), chỉ đếm một màu (`'B'`), vượt biên (`range(3)` cho góc trên-trái), **sao chép nông** làm hỏng lưới gốc, và chỉ kiểm tra một hình vuông.

## 2. Đề bài & ví dụ

Cho một ma trận `grid` kích thước `3 × 3` gồm các ký tự `'B'` và `'W'` (`'B'` là màu đen, `'W'` là màu trắng).

Bạn cần **đổi màu của tối đa một ô** (đổi `'B'` thành `'W'` hoặc `'W'` thành `'B'`, hoặc không đổi ô nào) sao cho **tồn tại một hình vuông `2 × 2` có cả bốn ô cùng màu**.

Trả về `true` nếu làm được, ngược lại `false`.

**Ví dụ 1**

- Input: `grid = [["B","W","B"],["B","W","W"],["B","W","B"]]`
- Output: `true`
- Giải thích: hình vuông `2 × 2` ở góc trên-phải gồm các ô `W, B, W, W`: ba ô `W` và một ô `B`. Đổi ô `B` thành `W` thì cả bốn ô cùng màu `W`.

**Ví dụ 2**

- Input: `grid = [["B","W","B"],["W","B","W"],["B","W","B"]]`
- Output: `false`
- Giải thích: lưới là bàn cờ vua; mọi hình vuông `2 × 2` đều có hai ô `B` và hai ô `W` nên cần đổi **hai** ô mới thành cùng màu, không thể chỉ đổi một ô.

**Ví dụ 3**

- Input: `grid = [["B","W","B"],["B","W","W"],["B","W","W"]]`
- Output: `true`
- Giải thích: hình vuông `2 × 2` ở góc dưới-phải (hàng `1..2`, cột `1..2`) đã có cả bốn ô `W` sẵn (không cần đổi gì).

**Ràng buộc:** `grid.length == 3`; `grid[i].length == 3`; `grid[i][j]` là `'W'` hoặc `'B'`.

In [1]:
from typing import List
from collections import Counter
import copy
import itertools
import random

examples = [
    ([["B", "W", "B"], ["B", "W", "W"], ["B", "W", "B"]], True),
    ([["B", "W", "B"], ["W", "B", "W"], ["B", "W", "B"]], False),
    ([["B", "W", "B"], ["B", "W", "W"], ["B", "W", "W"]], True),
]
for grid, expected in examples:
    print(grid, "-> kỳ vọng", expected)

[['B', 'W', 'B'], ['B', 'W', 'W'], ['B', 'W', 'B']] -> kỳ vọng True
[['B', 'W', 'B'], ['W', 'B', 'W'], ['B', 'W', 'B']] -> kỳ vọng False
[['B', 'W', 'B'], ['B', 'W', 'W'], ['B', 'W', 'W']] -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Ma trận 2 chiều: `grid[i][j]`

Ma trận là **list của các list**. `grid[i]` là hàng `i`, `grid[i][j]` là ô ở hàng `i`, cột `j` (chỉ số từ `0`). Lưới `3 × 3` có chỉ số `0, 1, 2` cho cả hàng và cột.

In [2]:
grid = [["B", "W", "B"], ["B", "W", "W"], ["B", "W", "B"]]
print(grid[0])          # ['B', 'W', 'B']   (hàng 0)
print(grid[1][2])       # 'W'               (hàng 1, cột 2)
print(len(grid), len(grid[0]))     # 3 3

['B', 'W', 'B']
W
3 3


### 3.2. Một hình vuông `2 × 2` được xác định bởi góc trên-trái `(r, c)`

Hình vuông `2 × 2` có góc trên-trái ở `(r, c)` gồm bốn ô: `(r, c)`, `(r, c + 1)`, `(r + 1, c)`, `(r + 1, c + 1)`. Để cả hình vuông nằm trong lưới `3 × 3`, `r` và `c` chỉ nhận giá trị `0` hoặc `1` (nếu `r = 2` thì `r + 1 = 3` vượt biên). Vậy có đúng **`2 × 2 = 4` hình vuông** cần xét.

In [3]:
grid = [["B", "W", "B"], ["B", "W", "W"], ["B", "W", "B"]]
for r in range(2):
    for c in range(2):
        cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
        print(f"góc trên-trái ({r}, {c}): {cells}")

góc trên-trái (0, 0): ['B', 'W', 'B', 'W']
góc trên-trái (0, 1): ['W', 'B', 'W', 'W']
góc trên-trái (1, 0): ['B', 'W', 'B', 'W']
góc trên-trái (1, 1): ['W', 'W', 'W', 'B']


### 3.3. `list.count` và `Counter`

`cells.count('B')` là số ô `B` trong list `cells`. Số ô `W` là `4 - cells.count('B')` (vì mỗi ô là `B` hoặc `W`). `Counter(cells)` cho cả hai số cùng lúc; `.most_common(1)` trả về cặp (màu, số lần) của màu xuất hiện nhiều nhất.

In [4]:
cells = ["W", "B", "W", "W"]
print(cells.count("B"), cells.count("W"))          # 1 3
c = Counter(cells)
print(c, c.most_common(1))                         # Counter({'W': 3, 'B': 1}) [('W', 3)]
print(max(c.values()))                             # 3

1 3
Counter({'W': 3, 'B': 1}) [('W', 3)]
3


### 3.4. Sao chép sâu (`copy.deepcopy`) khi thử sửa ma trận

`grid[:]` (hoặc `list(grid)`) chỉ sao chép danh sách **ngoài**, các **hàng** vẫn là cùng đối tượng; sửa `copy[0][0]` sẽ sửa luôn `grid[0][0]`. Muốn bản sao độc lập phải sao chép **từng hàng** (`[row[:] for row in grid]`) hoặc `copy.deepcopy(grid)`.

In [5]:
grid = [["B", "W"], ["W", "B"]]
shallow = grid[:]
shallow[0][0] = "X"
print(grid)                       # [['X', 'W'], ['W', 'B']]  <- gốc bị đổi!

grid = [["B", "W"], ["W", "B"]]
deep = [row[:] for row in grid]
deep[0][0] = "X"
print(grid, deep)                 # gốc giữ nguyên

[['X', 'W'], ['W', 'B']]
[['B', 'W'], ['W', 'B']] [['X', 'W'], ['W', 'B']]


### 3.5. Đảo màu một ô

Để đổi màu một ô, ta dùng phép "nếu là `'B'` thì thành `'W'`, ngược lại thành `'B'`": `"W" if x == "B" else "B"`.

In [6]:
flip = lambda x: "W" if x == "B" else "B"
print(flip("B"), flip("W"))        # W B

W B


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét mỗi hình vuông `2 × 2` (bốn ô) và hỏi: **cần đổi nhiều nhất bao nhiêu ô để cả bốn cùng màu?**

| số ô `B` / số ô `W` trong hình vuông | số ô phải đổi để cùng màu | làm được với ≤ 1 lần đổi? |
|---|---|---|
| 4 B / 0 W | 0 | **có** (đã cùng màu) |
| 3 B / 1 W | 1 (đổi ô `W` thành `B`) | **có** |
| 2 B / 2 W | 2 | không |
| 1 B / 3 W | 1 (đổi ô `B` thành `W`) | **có** |
| 0 B / 4 W | 0 | **có** |

Quy tắc: hình vuông "tốt" khi số ô `B` **khác `2`**. Đáp án `true` nếu **có ít nhất một** hình vuông tốt. Đáp án `false` khi **cả bốn** hình vuông đều là `2-2`.

Với ví dụ 2 (bàn cờ): cả `4` hình vuông đều `2 B / 2 W` ⇒ `false`.

In [7]:
grid = [["B", "W", "B"], ["W", "B", "W"], ["B", "W", "B"]]
for r in range(2):
    for c in range(2):
        cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
        print(f"({r}, {c}): {cells} -> số B = {cells.count('B')}")

(0, 0): ['B', 'W', 'W', 'B'] -> số B = 2
(0, 1): ['W', 'B', 'B', 'W'] -> số B = 2
(1, 0): ['W', 'B', 'B', 'W'] -> số B = 2
(1, 1): ['B', 'W', 'W', 'B'] -> số B = 2


## 5. Cách 1 — Brute Force / baseline: thử mọi phép đổi một ô rồi kiểm tra

**Ý tưởng.** Chỉ có `1 + 9 = 10` lựa chọn: **không đổi** hoặc **đổi một trong `9` ô**. Với mỗi lựa chọn: sao chép lưới, đổi ô (nếu có), rồi kiểm tra bốn hình vuông `2 × 2` xem có hình vuông cùng màu không. Nếu có ở bất kỳ lựa chọn nào ⇒ `true`.

Cần một hàm `has_square(g)` kiểm tra lưới `g` đã có hình vuông `2 × 2` cùng màu chưa.

In [8]:
def has_square(g: List[List[str]]) -> bool:
    for r in range(2):
        for c in range(2):
            if g[r][c] == g[r][c + 1] == g[r + 1][c] == g[r + 1][c + 1]:     # bốn ô cùng màu
                return True
    return False


def can_make_square_bruteforce(grid: List[List[str]]) -> bool:
    if has_square(grid):                                  # không cần đổi
        return True
    for i in range(3):
        for j in range(3):
            g = [row[:] for row in grid]                  # sao chép sâu để không làm hỏng grid gốc
            g[i][j] = "W" if g[i][j] == "B" else "B"      # đổi màu ô (i, j)
            if has_square(g):
                return True
    return False


for grid, expected in examples:
    print(can_make_square_bruteforce(grid), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


**Độ phức tạp (Cách 1).** `10` lựa chọn × (sao chép `9` ô + kiểm tra `4` hình vuông × `4` ô) ⇒ **Time `O(1)`** (hằng số, vì lưới cố định `3 × 3`), **Space `O(1)`**.

**Nút thắt:** việc sao chép và thử đổi từng ô là thừa — chỉ cần nhìn **tỉ lệ màu** trong mỗi hình vuông.

## 6. Ý tưởng thuật toán / bước đột phá

Mỗi hình vuông `2 × 2` **độc lập**. Để một hình vuông cùng màu chỉ cần đổi **số ô thuộc màu ít hơn**:

- màu ít hơn có `0` ô ⇒ đã cùng màu (đổi `0`);
- màu ít hơn có `1` ô ⇒ đổi `1` ô;
- màu ít hơn có `2` ô (`2-2`) ⇒ phải đổi `2` ô ⇒ không được.

Vậy hình vuông làm được ⇔ **một màu chiếm `>= 3` ô** ⇔ **số ô `B` thuộc `{0, 1, 3, 4}`** ⇔ **số ô `B` khác `2`**. Đáp án = có hình vuông nào như vậy hay không. Không cần sao chép hay thử đổi.

In [9]:
# Kiểm chứng: với mọi cấu hình 4 ô (2^4 = 16), "làm được bằng <= 1 lần đổi" <=> "số B khác 2"
for cells in itertools.product("BW", repeat=4):
    cells = list(cells)
    ok_by_flipping = (len(set(cells)) == 1) or any(
        len(set(cells[:k] + [("W" if cells[k] == "B" else "B")] + cells[k + 1:])) == 1 for k in range(4)
    )
    assert ok_by_flipping == (cells.count("B") != 2), cells
print("quy tắc 'số ô B khác 2' đúng trên mọi 16 cấu hình của một hình vuông")

quy tắc 'số ô B khác 2' đúng trên mọi 16 cấu hình của một hình vuông


## 7. Các cách tiếp theo

### Cách 2 — Đếm số ô `B` trong mỗi hình vuông

**Giải thích từng đoạn.**

1. Với `r` và `c` thuộc `{0, 1}`, lấy bốn ô: `grid[r][c]`, `grid[r][c+1]`, `grid[r+1][c]`, `grid[r+1][c+1]`.
2. Đếm số ô `'B'` (`count_b`).
3. Nếu `count_b != 2` thì hình vuông này sửa được (đổi tối đa `1` ô) ⇒ trả `True`.
4. Hết `4` hình vuông mà không có cái nào ⇒ `False`.

In [10]:
def can_make_square_count(grid: List[List[str]]) -> bool:
    for r in range(2):
        for c in range(2):
            cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
            if cells.count("B") != 2:                 # 0, 1, 3 hoặc 4 ô B: sửa được
                return True
    return False


for grid, expected in examples:
    print(can_make_square_count(grid), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionCount:
    def canMakeSquare(self, grid: List[List[str]]) -> bool:
        for r in range(2):
            for c in range(2):
                black = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]].count("B")
                if black != 2:
                    return True
        return False


print(SolutionCount().canMakeSquare([["B", "W", "B"], ["W", "B", "W"], ["B", "W", "B"]]))      # False

False


**Độ phức tạp Cách 2:** **Time `O(1)`** (`4` hình vuông × `4` ô), **Space `O(1)`**.

### Cách 3 — Đếm cả hai màu bằng `Counter`, lấy số lần lớn nhất `>= 3`

**Giải thích.** Thay vì "số `B` khác `2`", ta dùng cách phát biểu tự nhiên: **một màu chiếm ít nhất `3` ô**. `Counter` đếm cả `B` lẫn `W`, `max(counter.values())` là số lần của màu phổ biến nhất. Cách này không phụ thuộc "chỉ có hai màu" nếu sau này đề mở rộng (dễ sửa thành `>= 3` trên nhiều màu).

In [12]:
def can_make_square_counter(grid: List[List[str]]) -> bool:
    for r in range(2):
        for c in range(2):
            cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
            if max(Counter(cells).values()) >= 3:      # một màu chiếm >= 3 ô
                return True
    return False


for grid, expected in examples:
    print(can_make_square_counter(grid), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


**Độ phức tạp Cách 3:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 4 — Gọn bằng `any` và biểu thức sinh

**Giải thích.** Gộp hai vòng `for` bằng `itertools.product(range(2), repeat=2)` hoặc `for r in range(2) for c in range(2)` trong `any(...)`. Mỗi phần tử là điều kiện "số ô `B` khác `2`". Gọn nhất; nếu bạn mới học `any` thì đọc như "có hình vuông nào thỏa không".

In [13]:
def can_make_square_any(grid: List[List[str]]) -> bool:
    return any(
        [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]].count("B") != 2
        for r in range(2)
        for c in range(2)
    )


for grid, expected in examples:
    print(can_make_square_any(grid), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


**Độ phức tạp Cách 4:** **Time `O(1)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Ta trace Cách 2 trên ví dụ 1 và ví dụ 2.

In [14]:
def trace_count(grid: List[List[str]]) -> bool:
    print("grid =", grid)
    for r in range(2):
        for c in range(2):
            cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
            black = cells.count("B")
            ok = black != 2
            print(f"góc ({r}, {c}): ô = {cells}, số B = {black}, số W = {4 - black} -> {'sửa được (<= 1 lần đổi)' if ok else 'cần 2 lần đổi'}")
            if ok:
                print("=> True")
                return True
    print("=> False")
    return False


assert trace_count([["B", "W", "B"], ["B", "W", "W"], ["B", "W", "B"]]) is True
print()
assert trace_count([["B", "W", "B"], ["W", "B", "W"], ["B", "W", "B"]]) is False

grid = [['B', 'W', 'B'], ['B', 'W', 'W'], ['B', 'W', 'B']]
góc (0, 0): ô = ['B', 'W', 'B', 'W'], số B = 2, số W = 2 -> cần 2 lần đổi
góc (0, 1): ô = ['W', 'B', 'W', 'W'], số B = 1, số W = 3 -> sửa được (<= 1 lần đổi)
=> True

grid = [['B', 'W', 'B'], ['W', 'B', 'W'], ['B', 'W', 'B']]
góc (0, 0): ô = ['B', 'W', 'W', 'B'], số B = 2, số W = 2 -> cần 2 lần đổi
góc (0, 1): ô = ['W', 'B', 'B', 'W'], số B = 2, số W = 2 -> cần 2 lần đổi
góc (1, 0): ô = ['W', 'B', 'B', 'W'], số B = 2, số W = 2 -> cần 2 lần đổi
góc (1, 1): ô = ['B', 'W', 'W', 'B'], số B = 2, số W = 2 -> cần 2 lần đổi
=> False


Bảng trace Cách 2 với ví dụ 1 (`[["B","W","B"],["B","W","W"],["B","W","B"]]`):

| góc `(r, c)` | bốn ô | số B | `!= 2`? | kết quả |
|---|---|---|---|---|
| (0, 0) | B W B W | 2 | không | tiếp tục |
| (0, 1) | W B W W | 1 | **có** | `True` (đổi ô `B` thành `W`) |

Ví dụ 2 (bàn cờ): cả bốn hình vuông có `2 B` ⇒ không dừng sớm ⇒ `False`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (đếm số ô `B` trong mỗi hình vuông): ngắn, đúng, `O(1)`.

In [15]:
from typing import List

class Solution:
    def canMakeSquare(self, grid: List[List[str]]) -> bool:
        for r in range(2):                               # góc trên-trái của hình vuông 2x2: chỉ 0 hoặc 1
            for c in range(2):
                cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
                if cells.count("B") != 2:                # một màu chiếm >= 3 ô: đổi tối đa 1 ô là xong
                    return True
        return False


sol = Solution()
for grid, expected in examples:
    print(grid, "->", sol.canMakeSquare(grid))

[['B', 'W', 'B'], ['B', 'W', 'W'], ['B', 'W', 'B']] -> True
[['B', 'W', 'B'], ['W', 'B', 'W'], ['B', 'W', 'B']] -> False
[['B', 'W', 'B'], ['B', 'W', 'W'], ['B', 'W', 'W']] -> True


## 10. Vì sao thuật toán đúng

- **Chỉ có 4 hình vuông:** trong lưới `3 × 3`, hình vuông `2 × 2` có góc trên-trái `(r, c)` với `r, c ∈ {0, 1}`. Đổi một ô chỉ ảnh hưởng tới các hình vuông chứa ô đó; ta xét **từng** hình vuông riêng.
- **Một hình vuông sửa được ⇔ màu ít hơn có `<= 1` ô:** nếu màu ít hơn có `k` ô thì cần đúng `k` lần đổi (đổi từng ô đó sang màu kia) — hoặc đổi cả màu nhiều hơn (cần `4 - k >= k`, tệ hơn hoặc bằng). Vậy cần `min(k, 4 - k)` lần đổi, và `min <= 1 ⇔ k ∈ {0, 1, 3, 4}`.
- **Số `B` khác `2`:** vì tổng là `4`, `k` (số `B`) `∈ {0, 1, 3, 4}` ⇔ `k != 2`.
- **Kết luận:** có thể làm được ⇔ **tồn tại** hình vuông với số `B` khác `2`. Bảng trên được kiểm chứng vét cạn trên cả `16` cấu hình (§6).

## 11. Phân tích Time / Space Complexity

Lưới cố định `3 × 3`, nên mọi cách đều `O(1)`. Số bước tương đối:

| Cách | Số bước (tương đối) | Space |
|---|---|---|
| 1. Thử mọi phép đổi | `~10 × (9 + 16)` | `O(1)` |
| 2. Đếm số ô `B` | `~4 × 4` | `O(1)` |
| 3. `Counter` | `~4 × 4` | `O(1)` |
| 4. `any` gọn | `~4 × 4` | `O(1)` |

Nếu lưới tổng quát `m × n` và hình vuông `k × k` thì đếm theo hình vuông cần `O(m · n · k²)` hoặc `O(m · n)` với tổng cộng dồn 2D.

## 12. Bộ test

Ta gom các cách vào `impls`; Cách 1 là chuẩn bám sát đề.

In [16]:
sol = Solution()
impls = {
    "brute": can_make_square_bruteforce,
    "count": can_make_square_count,
    "counter": can_make_square_counter,
    "any": can_make_square_any,
    "leetcode": sol.canMakeSquare,
}


def check_all(grid, expected, label):
    for key, fn in impls.items():
        g = copy.deepcopy(grid)
        actual = fn(g)
        assert actual == expected, (label, key, grid, actual, expected)
        assert g == grid, ("input bị sửa", key)
    print("PASS", label)


for idx, (grid, expected) in enumerate(examples, 1):
    check_all(grid, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [17]:
check_all([["B"] * 3 for _ in range(3)], True, "toàn B: đã có hình vuông")
check_all([["W"] * 3 for _ in range(3)], True, "toàn W")
check_all([["B", "W", "B"], ["W", "B", "W"], ["B", "W", "B"]], False, "bàn cờ B ở góc")
check_all([["W", "B", "W"], ["B", "W", "B"], ["W", "B", "W"]], False, "bàn cờ W ở góc")
check_all([["B", "B", "W"], ["W", "W", "B"], ["B", "B", "W"]], False, "mọi hình vuông đều 2 B / 2 W")
check_all([["B", "B", "W"], ["B", "W", "W"], ["W", "W", "W"]], True, "hình vuông dưới-phải toàn W")
check_all([["B", "W", "W"], ["W", "W", "W"], ["B", "B", "B"]], True, "hình vuông trên-phải toàn W")
check_all([["B", "B", "B"], ["W", "W", "W"], ["B", "B", "B"]], False, "ba hàng xen kẽ: mỗi hình vuông 2B 2W")
check_all([["B", "W", "B"], ["B", "W", "B"], ["B", "W", "B"]], False, "ba cột xen kẽ: mỗi hình vuông 2B 2W")
check_all([["B", "B", "W"], ["B", "W", "W"], ["W", "W", "B"]], True, "hình vuông có 3 W 1 B")

PASS toàn B: đã có hình vuông
PASS toàn W
PASS bàn cờ B ở góc
PASS bàn cờ W ở góc
PASS mọi hình vuông đều 2 B / 2 W
PASS hình vuông dưới-phải toàn W
PASS hình vuông trên-phải toàn W
PASS ba hàng xen kẽ: mỗi hình vuông 2B 2W
PASS ba cột xen kẽ: mỗi hình vuông 2B 2W
PASS hình vuông có 3 W 1 B


Ngoài các ca chọn tay trên, mục 12.3 sẽ **vét cạn mọi lưới `3 × 3`** (`2^9 = 512` lưới) để chắc chắn không bỏ sót ca nào.

### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi phổ biến: (a) **chỉ xét hình vuông có đúng `3` ô cùng màu** (bỏ sót `4` ô sẵn có); (b) **chỉ đếm màu `'B'` `>= 3`** (bỏ sót `1 B + 3 W`); (c) **chỉ kiểm tra hình vuông ở góc trên-trái**.

In [18]:
def wrong_exactly_three(grid):
    for r in range(2):
        for c in range(2):
            cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
            if cells.count("B") == 3 or cells.count("W") == 3:       # SAI: bỏ sót 4 ô cùng màu
                return True
    return False


def wrong_only_black(grid):
    for r in range(2):
        for c in range(2):
            cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
            if cells.count("B") >= 3:                                # SAI: quên màu W
                return True
    return False


def wrong_only_first_square(grid):
    cells = [grid[0][0], grid[0][1], grid[1][0], grid[1][1]]
    return cells.count("B") != 2                                     # SAI: chỉ một hình vuông


all_black = [["B"] * 3 for _ in range(3)]
mostly_white = [["B", "W", "W"], ["W", "W", "W"], ["B", "B", "B"]]
ex1 = examples[0][0]
print(wrong_exactly_three(all_black), "!=", can_make_square_count(all_black))
print(wrong_only_black(mostly_white), "!=", can_make_square_count(mostly_white))
print(wrong_only_first_square(ex1), "!=", can_make_square_count(ex1))
assert wrong_exactly_three(all_black) != can_make_square_count(all_black)
assert wrong_only_black(mostly_white) != can_make_square_count(mostly_white)
assert wrong_only_first_square(ex1) != can_make_square_count(ex1)

False != True
False != True
False != True


### 12.3. Vét cạn mọi lưới `3 × 3` (`2^9 = 512` lưới)

Đây là cách kiểm chứng mạnh nhất: **mọi** đầu vào hợp lệ chỉ có `512`. Cách 1 (thử mọi phép đổi) là chuẩn; mọi cách khác phải khớp trên **cả 512 lưới**. Ngoài ra ta kiểm tra thêm thuộc tính: đáp án `False` **chỉ khi** cả bốn hình vuông đều `2 B / 2 W`.

In [19]:
count_true = count_false = 0
for bits in itertools.product("BW", repeat=9):
    grid = [list(bits[0:3]), list(bits[3:6]), list(bits[6:9])]
    ref = can_make_square_bruteforce(copy.deepcopy(grid))
    for key, fn in impls.items():
        assert fn(copy.deepcopy(grid)) == ref, (key, grid)
    # thuộc tính: False <=> cả 4 hình vuông có đúng 2 ô B
    all_two = all(
        [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]].count("B") == 2
        for r in range(2) for c in range(2)
    )
    assert ref == (not all_two), grid
    count_true += ref
    count_false += not ref
print("PASS toàn bộ 512 lưới; số lưới True =", count_true, ", False =", count_false)
assert count_true + count_false == 512 and count_false > 0

PASS toàn bộ 512 lưới; số lưới True = 498 , False = 14


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ xét hình vuông có đúng `3` ô cùng màu

In [20]:
def sai_chi_ba(grid):
    for r in range(2):
        for c in range(2):
            cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
            if cells.count("B") == 3 or cells.count("W") == 3:       # SAI: bỏ sót 4 ô sẵn
                return True
    return False


grid = [["B"] * 3 for _ in range(3)]
print("sai:", sai_chi_ba(grid), "| đúng:", Solution().canMakeSquare(grid))

sai: False | đúng: True


**Vì sao sai:** nếu một hình vuông đã có cả `4` ô cùng màu thì **không cần đổi** (đề cho phép "tối đa một ô", gồm cả `0`) ⇒ phải trả `true`. **Cách sửa:** điều kiện "số `B` khác `2`" (hoặc "một màu `>= 3` ô").

### Lỗi 2: Chỉ đếm màu `'B'`

In [21]:
def sai_chi_dem_B(grid):
    for r in range(2):
        for c in range(2):
            cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
            if cells.count("B") >= 3:                                # SAI: quên 1 B + 3 W
                return True
    return False


grid = [["B", "W", "W"], ["W", "W", "W"], ["B", "B", "B"]]
print("sai:", sai_chi_dem_B(grid), "| đúng:", Solution().canMakeSquare(grid))

sai: False | đúng: True


**Vì sao sai:** hình vuông `W W / W W` (hoặc `B` đơn lẻ giữa ba `W`) cũng sửa được. Mất đối xứng giữa hai màu. **Cách sửa:** xét cả hai màu, hoặc dùng "số `B` khác `2`".

### Lỗi 3: Vượt biên khi duyệt góc trên-trái

In [22]:
def sai_vuot_bien(grid):
    for r in range(3):                              # SAI: r = 2 thì r + 1 = 3 vượt biên
        for c in range(3):
            cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
            if cells.count("B") != 2:
                return True
    return False


try:
    sai_vuot_bien([["B", "W", "B"], ["W", "B", "W"], ["B", "W", "B"]])
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().canMakeSquare([["B", "W", "B"], ["W", "B", "W"], ["B", "W", "B"]]))

IndexError: list index out of range
đúng: False


**Vì sao sai:** góc trên-trái của hình vuông `2 × 2` chỉ có thể ở `r, c ∈ {0, 1}`. Với `r = 2` hoặc `c = 2`, `grid[r + 1]` / `[c + 1]` vượt biên (`IndexError`). **Cách sửa:** `range(2)` (tức `range(len(grid) - 1)`).

### Lỗi 4: Sao chép nông khi thử đổi màu (làm hỏng lưới gốc)

In [23]:
def sai_sao_chep_nong(grid):
    for i in range(3):
        for j in range(3):
            g = grid[:]                              # SAI: sao chép nông, các hàng vẫn dùng chung
            g[i][j] = "W" if g[i][j] == "B" else "B"
            if has_square(g):
                return True
    return False


grid = [["B", "B", "B"], ["W", "W", "W"], ["B", "B", "B"]]     # ba hàng xen kẽ: đúng là False
before = copy.deepcopy(grid)
print("sai:", sai_sao_chep_nong(grid), "| đúng:", Solution().canMakeSquare(before))
print("grid gốc đã bị sửa?", grid != before)

sai: True | đúng: False
grid gốc đã bị sửa? True


**Vì sao sai:** `grid[:]` chỉ sao chép danh sách ngoài, các hàng vẫn là cùng đối tượng. Mỗi lần "thử" đổi một ô sẽ **thật sự đổi ô đó trong lưới gốc** và không bao giờ hoàn tác ⇒ các lần thử sau chồng lên nhau, có thể ra `true` sai (ở ví dụ trên, sau khi đổi liên tiếp nhiều ô của hàng giữa, lưới "vô tình" thành có hình vuông cùng màu). **Cách sửa:** sao chép từng hàng (`[row[:] for row in grid]`) hoặc `copy.deepcopy`, hoặc tốt hơn: dùng cách đếm không cần sửa lưới.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Miền **cực nhỏ và cố định** (`3 × 3`, hai màu): có thể **vét cạn mọi đầu vào** (`512`) để kiểm chứng; hãy tận dụng.
- "Đổi tối đa `k` ô để đạt tính chất cục bộ" ⇒ xét **từng vùng cục bộ độc lập**; số lần đổi tối thiểu = số ô thuộc màu ít hơn (khi chỉ có hai màu).
- Hai màu, vùng `4` ô: điều kiện `min(số B, số W) <= 1` hay "số B khác `2`".
- Cẩn thận sao chép lưới: **sao chép sâu** mới an toàn khi thử sửa.
- Với bài lưới lớn `m × n`: dùng tổng cộng dồn 2D hoặc trượt cửa sổ `2 × 2` để đếm `B` trong `O(1)` mỗi ô.

## 15. Biến thể của bài

1. **Lưới `m × n` bất kỳ, hình vuông `2 × 2`, đổi tối đa một ô.**
2. **Số lần đổi ít nhất** để có hình vuông `2 × 2` cùng màu (`0`, `1` hoặc `2`).
3. **Đếm số hình vuông `2 × 2` đã cùng màu** (không đổi).
4. **Hình vuông `k × k`** (tổng quát).
5. **Count Square Submatrices with All Ones (1277)**, **Maximal Square (221)**.

In [24]:
def min_flips_to_square(grid):
    best = 4
    m, n = len(grid), len(grid[0])
    for r in range(m - 1):
        for c in range(n - 1):
            cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
            b = cells.count("B")
            best = min(best, b, 4 - b)
    return best


def count_ready_squares(grid):
    m, n = len(grid), len(grid[0])
    return sum(
        grid[r][c] == grid[r][c + 1] == grid[r + 1][c] == grid[r + 1][c + 1]
        for r in range(m - 1) for c in range(n - 1)
    )


def can_make_square_general(grid):
    return min_flips_to_square(grid) <= 1


print(min_flips_to_square(examples[0][0]), min_flips_to_square(examples[1][0]))      # 1 2
print(count_ready_squares(examples[2][0]))                                            # 1
assert min_flips_to_square(examples[0][0]) == 1
assert min_flips_to_square(examples[1][0]) == 2
assert count_ready_squares(examples[2][0]) == 1
assert can_make_square_general([["B", "W", "B", "W"], ["W", "B", "W", "B"], ["B", "B", "W", "B"]]) is True

1 2
1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Số bước | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi phép đổi | sao chép, đổi, kiểm tra | `~250` | `O(1)` | bám sát đề, làm chuẩn kiểm tra |
| 2. Đếm số ô `B` | `count("B") != 2` | `~16` | `O(1)` | **nên nộp** |
| 3. `Counter` | một màu chiếm `>= 3` | `~16` | `O(1)` | khi muốn nói đúng ý "màu chiếm đa số" |
| 4. `any` gọn | một biểu thức | `~16` | `O(1)` | khi thích viết ngắn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `["B", "W", "W", "W"].count("W")`, `[["B", "W"], ["W", "B"]][1][0]`, `list(range(2))`, `Solution().canMakeSquare([["B"] * 3, ["B"] * 3, ["W"] * 3])`, `Solution().canMakeSquare([["B", "B", "W"], ["W", "W", "B"], ["B", "B", "W"]])`.

**Bài 2 (tính tay).** Với `grid = [["B","B","W"],["W","W","B"],["B","B","W"]]`: viết ra bốn hình vuông `2 × 2`, số ô `B` của mỗi hình vuông và đáp án.

**Bài 3 (điền TODO).** Hoàn thành `can_make_square_todo(grid)` theo Cách 2.

In [25]:
def can_make_square_todo(grid):
    # TODO 1: duyệt r trong range(2) và c trong range(2) (góc trên-trái của hình vuông 2x2)
    #   TODO 2: tạo list cells gồm 4 ô: (r, c), (r, c+1), (r+1, c), (r+1, c+1)
    #   TODO 3: nếu cells.count("B") != 2 thì return True
    # TODO 4: nếu không hình vuông nào thỏa, return False
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `can_make_square_flip(grid)` theo kiểu **thử đổi từng ô** nhưng **không sao chép lưới**: đổi ô, kiểm tra, rồi **đổi lại** ô đó (hoàn tác) trước khi thử ô khác.

**Bài 5 (biến thể / mini challenge).** (a) Viết `min_flips_any_size(grid)` với lưới `m × n` bất kỳ (`m, n >= 2`): số lần đổi ít nhất để có một hình vuông `2 × 2` cùng màu. (b) Viết `count_fixable_squares(grid)`: đếm số hình vuông `2 × 2` (theo góc trên-trái) mà **sửa được bằng tối đa một lần đổi**.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `3`, `'W'`, `[0, 1]`, `True` (hàng 0-1 toàn B), `False`. Bài 3: nhớ `range(2)` (không phải `range(3)`).
- **Mức 2 (vừa):** Bài 2: bốn hình vuông: `(0,0)`: B B W W (2 B); `(0,1)`: B W W B (2 B); `(1,0)`: W W B B (2 B); `(1,1)`: W B B W (2 B) ⇒ tất cả `2` ⇒ `False`. Bài 4: `g[i][j] = flip(g[i][j])`, kiểm tra `has_square(g)`, rồi đổi lại.
- **Mức 3 (sâu):** Bài 5(a): `min(b, 4 - b)` cho mỗi hình vuông, lấy `min` trên mọi hình vuông (`0`, `1` hoặc `2`). 5(b): đếm hình vuông có `min(b, 4 - b) <= 1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1
assert ["B", "W", "W", "W"].count("W") == 3
assert [["B", "W"], ["W", "B"]][1][0] == "W"
assert list(range(2)) == [0, 1]
assert Solution().canMakeSquare([["B"] * 3, ["B"] * 3, ["W"] * 3]) is True
assert Solution().canMakeSquare([["B", "B", "W"], ["W", "W", "B"], ["B", "B", "W"]]) is False

# Bài 2
g2 = [["B", "B", "W"], ["W", "W", "B"], ["B", "B", "W"]]
counts = []
for r in range(2):
    for c in range(2):
        counts.append([g2[r][c], g2[r][c + 1], g2[r + 1][c], g2[r + 1][c + 1]].count("B"))
assert counts == [2, 2, 2, 2]
assert Solution().canMakeSquare(g2) is False


# Bài 3
def can_make_square_todo_ans(grid):
    for r in range(2):
        for c in range(2):
            cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
            if cells.count("B") != 2:
                return True
    return False


for grid, expected in examples:
    assert can_make_square_todo_ans(grid) == expected


# Bài 4: đổi tại chỗ rồi hoàn tác (không sao chép lưới)
def can_make_square_flip(grid):
    if has_square(grid):
        return True
    for i in range(3):
        for j in range(3):
            old = grid[i][j]
            grid[i][j] = "W" if old == "B" else "B"
            found = has_square(grid)
            grid[i][j] = old                         # hoàn tác
            if found:
                return True
    return False


for grid, expected in examples:
    g = copy.deepcopy(grid)
    assert can_make_square_flip(g) == expected
    assert g == grid


# Bài 5(a)
def min_flips_any_size(grid):
    m, n = len(grid), len(grid[0])
    best = 4
    for r in range(m - 1):
        for c in range(n - 1):
            b = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]].count("B")
            best = min(best, b, 4 - b)
    return best


assert min_flips_any_size(examples[0][0]) == 1
assert min_flips_any_size(examples[1][0]) == 2
assert min_flips_any_size([["B", "B", "B", "B"], ["B", "B", "B", "B"]]) == 0


# Bài 5(b)
def count_fixable_squares(grid):
    m, n = len(grid), len(grid[0])
    total = 0
    for r in range(m - 1):
        for c in range(n - 1):
            b = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]].count("B")
            total += min(b, 4 - b) <= 1
    return total


assert count_fixable_squares(examples[0][0]) == 2       # góc (0,1): 1 B + 3 W; góc (1,1): 3 W + 1 B
for bits in itertools.product("BW", repeat=9):
    grid = [list(bits[0:3]), list(bits[3:6]), list(bits[6:9])]
    assert (count_fixable_squares(grid) > 0) == Solution().canMakeSquare(grid)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (vùng `2 × 2` hai màu: sửa được khi một màu chiếm `>= 3` ô)

```python
from typing import List

class Solution:
    def canMakeSquare(self, grid: List[List[str]]) -> bool:
        for r in range(2):
            for c in range(2):
                cells = [grid[r][c], grid[r][c + 1], grid[r + 1][c], grid[r + 1][c + 1]]
                if cells.count("B") != 2:
                    return True
        return False
```

**Checklist**

- [ ] Chỉ xét `4` hình vuông (`r, c ∈ {0, 1}`), không vượt biên?
- [ ] Điều kiện là "số `B` khác `2`" (gồm cả `0`, `1`, `3`, `4`)?
- [ ] Đã xét cả hai màu (không chỉ `'B'`)?
- [ ] Nếu thử đổi ô: dùng sao chép **sâu** (hoặc hoàn tác) để không phá lưới gốc?
- [ ] Đã thử lưới toàn `B`, bàn cờ vua, và một hình vuông `3 W + 1 B`?
- [ ] Skeleton đúng `class Solution` / `def canMakeSquare(self, grid: List[List[str]]) -> bool`?